## Explore SRoBERTa Embeddings of IEMOCAP Data for Cross-Modal Attention Fusion

### Imports and paths

In [8]:
import numpy as np
import os
from pathlib import Path

embeddings_dir = Path("../../../data/embeddings/4way/sroberta/avg_last4/wmean_pos_rev")
files = {
    "train": embeddings_dir / "train_filtered.npz",
    "val": embeddings_dir / "val_filtered.npz",
    "test": embeddings_dir / "test_filtered.npz"
}

### Load and explore each file

In [9]:
sroberta_data = {}
for split, filepath in files.items():
    print(f"\n{'='*60}")
    print(f"Exploring {split.upper()} split: {filepath.name}")
    print(f"{'='*60}")
    
    if not filepath.exists():
        print(f"⚠️  File not found: {filepath}")
        continue
    
    data = np.load(filepath, allow_pickle=True)
    sroberta_data[split] = data
    
    # Print keys in the npz file
    print(f"Keys in NPZ: {list(data.files)}")
    print(f"Number of keys: {len(data.files)}")
    
    # Explore each array
    for key in data.files:
        arr = data[key]
        print(f"\n  Key: '{key}'")
        print(f"    Shape: {arr.shape}")
        print(f"    Dtype: {arr.dtype}")
        if arr.size > 0:
            print(f"    Min: {arr.min():.6f}, Max: {arr.max():.6f}, Mean: {arr.mean():.6f}")
        if len(arr.shape) > 1:
            print(f"    Dimensions: batch={arr.shape[0]}, features={arr.shape[1]}")
            if len(arr.shape) > 2:
                print(f"    Additional dims: {arr.shape[2:]}")


Exploring TRAIN split: train_filtered.npz
Keys in NPZ: ['embeddings']
Number of keys: 1

  Key: 'embeddings'
    Shape: (5766, 768)
    Dtype: float32
    Min: -5.761981, Max: 20.738029, Mean: 0.035625
    Dimensions: batch=5766, features=768

Exploring VAL split: val_filtered.npz
Keys in NPZ: ['embeddings']
Number of keys: 1

  Key: 'embeddings'
    Shape: (2103, 768)
    Dtype: float32
    Min: -5.815454, Max: 20.330011, Mean: 0.035522
    Dimensions: batch=2103, features=768

Exploring TEST split: test_filtered.npz
Keys in NPZ: ['embeddings']
Number of keys: 1

  Key: 'embeddings'
    Shape: (2170, 768)
    Dtype: float32
    Min: -5.965002, Max: 20.448303, Mean: 0.035639
    Dimensions: batch=2170, features=768


### Analysis for cross-modal attention

In [10]:
print("="*60)
print("ANALYSIS FOR CROSS-MODAL ATTENTION FUSION")
print("="*60)

splits_summary = {}
for split in ["train", "val", "test"]:
    if split not in sroberta_data:
        continue
    data = sroberta_data[split]
    # Try to find the embeddings/features array
    if 'embeddings' in data.files:
        features = data['embeddings']
    elif 'features' in data.files:
        features = data['features']
    else:
        # Use the first (and likely only) array
        features = data[data.files[0]]
    
    splits_summary[split] = {
        "num_samples": features.shape[0],
        "feature_dim": features.shape[1] if len(features.shape) > 1 else 1,
        "dtype": features.dtype
    }

print("\n1. FEATURE DIMENSIONS:")
for split, info in splits_summary.items():
    print(f"   {split.upper():8} -> Samples: {info['num_samples']:4d}, SRoBERTa Dim: {info['feature_dim']:4d}, Dtype: {info['dtype']}")

print("\n2. DATA TYPE & PRECISION:")
print(f"   SRoBERTa uses float32 (32-bit floating point)")
print(f"   This is standard for transformer embeddings")

print("\n3. SCALE & NORMALIZATION:")
for split in ["train", "val", "test"]:
    if split not in sroberta_data:
        continue
    data = sroberta_data[split]
    if 'embeddings' in data.files:
        features = data['embeddings']
    elif 'features' in data.files:
        features = data['features']
    else:
        features = data[data.files[0]]
    print(f"   {split.upper():8} -> Range: [{features.min():.4f}, {features.max():.4f}], Std: {features.std():.6f}")

print("\n4. KEY CONSIDERATIONS FOR CROSS-MODAL ATTENTION:")
if 'train' in splits_summary:
    sroberta_dim = splits_summary['train']['feature_dim']
    print(f"   ✓ SRoBERTa Feature Dimension: {sroberta_dim}")
    print(f"   ✓ Number of Utterances (samples): Train={splits_summary['train']['num_samples']}, " +
          f"Val={splits_summary.get('val', {}).get('num_samples', 'N/A')}, " +
          f"Test={splits_summary.get('test', {}).get('num_samples', 'N/A')}")
    print(f"   ✓ Data is typically normalized")
    print(f"   ✓ Consistent feature space across splits")
    print(f"\n   FUSION ARCHITECTURE NOTES:")
    print(f"   - SRoBERTa provides semantic/textual embeddings")
    print(f"   - Pair with HuBERT (audio, typically 1024 dims) for cross-modal fusion")
    print(f"   - If SRoBERTa is {sroberta_dim}D and HuBERT is 1024D:")
    if sroberta_dim == 1024:
        print(f"     → Direct concatenation possible: 2048 total dims")
    else:
        print(f"     → May want to project to common dim for attention: e.g., {max(sroberta_dim, 1024)}")
    print(f"   - Cross-attention: Query from one modality, Key/Value from another")
    print(f"   - Both embeddings must be aligned by utterance ID")
    print(f"   - Typical: project both to same hidden_dim before attention (e.g., 256-512)")

print("\nSUMMARY:")
print(" - SRoBERTa embeddings loaded and analyzed.")
print(" - Ready for concatenation or cross-attention fusion with audio embeddings (HuBERT).")

ANALYSIS FOR CROSS-MODAL ATTENTION FUSION

1. FEATURE DIMENSIONS:
   TRAIN    -> Samples: 5766, SRoBERTa Dim:  768, Dtype: float32
   VAL      -> Samples: 2103, SRoBERTa Dim:  768, Dtype: float32
   TEST     -> Samples: 2170, SRoBERTa Dim:  768, Dtype: float32

2. DATA TYPE & PRECISION:
   SRoBERTa uses float32 (32-bit floating point)
   This is standard for transformer embeddings

3. SCALE & NORMALIZATION:
   TRAIN    -> Range: [-5.7620, 20.7380], Std: 0.691299
   VAL      -> Range: [-5.8155, 20.3300], Std: 0.688318
   TEST     -> Range: [-5.9650, 20.4483], Std: 0.691398

4. KEY CONSIDERATIONS FOR CROSS-MODAL ATTENTION:
   ✓ SRoBERTa Feature Dimension: 768
   ✓ Number of Utterances (samples): Train=5766, Val=2103, Test=2170
   ✓ Data is typically normalized
   ✓ Consistent feature space across splits

   FUSION ARCHITECTURE NOTES:
   - SRoBERTa provides semantic/textual embeddings
   - Pair with HuBERT (audio, typically 1024 dims) for cross-modal fusion
   - If SRoBERTa is 768D and Hu